In [1]:
import os, re, math, random
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [2]:
data_dir = Path("data_manythings")
candidates = [
    data_dir / "ukr.txt",
    data_dir / "ukr-eng" / "ukr.txt",
    data_dir / "ukr-eng.txt",
]
txt_path = next((p for p in candidates if p.exists()), None)
if txt_path is None:
    raise FileNotFoundError("Put ManyThings file to data_manythings/ukr.txt")
lines = txt_path.read_text(encoding="utf-8").splitlines()
pairs = []
for l in lines:
    parts = l.split("\t")
    if len(parts) >= 2:
        a = parts[0].strip()
        b = parts[1].strip()
        if a and b:
            pairs.append((a, b))
print(txt_path.as_posix(), len(pairs), pairs[0])


data_manythings/ukr.txt 160049 ('Go.', 'Йди.')


In [3]:
def has_cyrillic(s):
    return bool(re.search(r"[А-Яа-яІіЇїЄєҐґ]", s))

sample = random.sample(pairs, k=min(2000, len(pairs)))
c0 = sum(has_cyrillic(p[0]) for p in sample) / len(sample)
c1 = sum(has_cyrillic(p[1]) for p in sample) / len(sample)
if c0 >= c1:
    ukr_eng = [(p[0], p[1]) for p in pairs]
else:
    ukr_eng = [(p[1], p[0]) for p in pairs]
random.shuffle(ukr_eng)
ukr_eng = ukr_eng[:60000]
print(ukr_eng[0])


('Я переклав цей лист на французьку для Тома.', 'I translated the letter into French for Tom.')


In [4]:
MAX_VOCAB = 15000
MAX_LEN_SRC = 20
MAX_LEN_TGT = 20
specials = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

def tokenize(text):
    return text.lower().strip().split()

def build_vocab(sentences):
    counter = {}
    for s in sentences:
        for w in tokenize(s):
            counter[w] = counter.get(w, 0) + 1
    vocab = specials + [w for w, _ in sorted(counter.items(), key=lambda x: x[1], reverse=True)[: MAX_VOCAB - len(specials)]]
    stoi = {w:i for i,w in enumerate(vocab)}
    itos = vocab
    return stoi, itos

ukr_sent = [p[0] for p in ukr_eng]
eng_sent = [p[1] for p in ukr_eng]

ukr_stoi, ukr_itos = build_vocab(ukr_sent)
eng_stoi, eng_itos = build_vocab(eng_sent)

PAD_IDX = ukr_stoi["[PAD]"]
UNK_IDX = ukr_stoi["[UNK]"]
BOS_IDX = ukr_stoi["[BOS]"]
EOS_IDX = ukr_stoi["[EOS]"]

print(len(ukr_stoi), len(eng_stoi))


15000 13364


In [5]:
def encode(sentence, stoi, max_len, add_specials=True):
    ids = []
    if add_specials:
        ids.append(BOS_IDX)
    for w in tokenize(sentence)[: max_len - (2 if add_specials else 0)]:
        ids.append(stoi.get(w, UNK_IDX))
    if add_specials:
        ids.append(EOS_IDX)
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

def prepare_pair(src_ukr, tgt_eng):
    src = encode(src_ukr, ukr_stoi, MAX_LEN_SRC, add_specials=True)
    tgt_in = encode(tgt_eng, eng_stoi, MAX_LEN_TGT, add_specials=True)
    tgt_out = tgt_in[1:] + [PAD_IDX]
    return src, tgt_in, tgt_out

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        src_ukr, tgt_eng = self.pairs[idx]
        src, tgt_in, tgt_out = prepare_pair(src_ukr, tgt_eng)
        return torch.tensor(src), torch.tensor(tgt_in), torch.tensor(tgt_out)

val_size = max(2000, int(0.05 * len(ukr_eng)))
train_pairs = ukr_eng[:-val_size]
val_pairs = ukr_eng[-val_size:]

batch_size = 64
train_loader = DataLoader(TranslationDataset(train_pairs), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TranslationDataset(val_pairs), batch_size=batch_size, shuffle=False)

x = next(iter(train_loader))
print(x[0].shape, x[1].shape, x[2].shape)


torch.Size([64, 20]) torch.Size([64, 20]) torch.Size([64, 20])


In [6]:
def make_padding_mask(x, pad_idx=PAD_IDX):
    return (x == pad_idx)

def generate_subsequent_mask(size, device):
    return torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)

class TokenPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=50, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        b, t = x.shape
        pos = torch.arange(t, device=x.device).unsqueeze(0).expand(b, t)
        return self.dropout(self.token_emb(x) + self.pos_emb(pos))

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, n_heads=4, num_layers=2, d_ff=128, max_len_src=20, max_len_tgt=20, dropout=0.1):
        super().__init__()
        self.src_emb = TokenPositionalEmbedding(src_vocab_size, d_model, max_len_src, dropout)
        self.tgt_emb = TokenPositionalEmbedding(tgt_vocab_size, d_model, max_len_tgt, dropout)
        self.transformer = nn.Transformer(d_model=d_model, nhead=n_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers, dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
    def forward(self, src, tgt_in):
        src_key_padding_mask = make_padding_mask(src)
        tgt_key_padding_mask = make_padding_mask(tgt_in)
        tgt_mask = generate_subsequent_mask(tgt_in.size(1), tgt_in.device)
        src_emb = self.src_emb(src)
        tgt_emb = self.tgt_emb(tgt_in)
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask, src_key_padding_mask=src_key_padding_mask, tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=src_key_padding_mask)
        return self.fc_out(out)

model = Seq2SeqTransformer(len(ukr_stoi), len(eng_stoi), d_model=64, n_heads=4, num_layers=2, d_ff=128, max_len_src=MAX_LEN_SRC, max_len_tgt=MAX_LEN_TGT).to(device)
print(sum(p.numel() for p in model.parameters()))


2854196


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_tokens = 0
    for src, tgt_in, tgt_out in loader:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(src, tgt_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        n = (tgt_out != PAD_IDX).sum().item()
        total_loss += loss.item() * n
        total_tokens += n
    return total_loss / max(1, total_tokens)

EPOCHS = 6
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    vl = run_epoch(val_loader, False)
    print(ep, tr, vl)


In [ ]:
def decode(ids, itos):
    out = []
    for i in ids:
        if i == EOS_IDX:
            break
        if i == PAD_IDX or i == BOS_IDX:
            continue
        if 0 <= i < len(itos):
            out.append(itos[i])
    return " ".join(out)

@torch.no_grad()
def translate(sentence_ukr, max_len=MAX_LEN_TGT):
    model.eval()
    src = torch.tensor([encode(sentence_ukr, ukr_stoi, MAX_LEN_SRC, add_specials=True)], dtype=torch.long).to(device)
    tgt_ids = [BOS_IDX]
    for _ in range(max_len - 1):
        tgt_in = torch.tensor([tgt_ids + [PAD_IDX] * (MAX_LEN_TGT - len(tgt_ids))], dtype=torch.long).to(device)
        logits = model(src, tgt_in)
        next_id = int(torch.argmax(logits[0, len(tgt_ids) - 1]).item())
        tgt_ids.append(next_id)
        if next_id == EOS_IDX:
            break
    return decode(tgt_ids, eng_itos)

for _ in range(5):
    s, ref = random.choice(ukr_eng)
    print(s)
    print(ref)
    print(translate(s))
    print()


In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

root = Path("./data")
raw = root / "FashionMNIST" / "raw"
needed = [
    raw / "train-images-idx3-ubyte.gz",
    raw / "train-labels-idx1-ubyte.gz",
    raw / "t10k-images-idx3-ubyte.gz",
    raw / "t10k-labels-idx1-ubyte.gz",
]
missing = [p.as_posix() for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError("Missing FashionMNIST raw files: " + "; ".join(missing))

batch_size = 128
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root=str(root), train=True, download=False, transform=transform)
test_dataset = datasets.FashionMNIST(root=str(root), train=False, download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(len(train_dataset), len(test_dataset))


In [ ]:
original_dim = 28 * 28
intermediate_dim = 512
latent_dim = 2
epochs = 15

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mean(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    def forward(self, z):
        h = F.relu(self.fc1(z))
        return torch.sigmoid(self.fc2(h))

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)
    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std
    def forward(self, x):
        mean, logvar = self.encoder(x)
        z = self.reparameterize(mean, logvar)
        recon = self.decoder(z)
        return recon, mean, logvar

def loss_fn(x_recon, x, z_mean, z_logvar):
    recon = F.binary_cross_entropy(x_recon, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + z_logvar - z_mean.pow(2) - z_logvar.exp(), dim=1).mean()
    return recon + kl

vae = VAE(original_dim, intermediate_dim, latent_dim).to(device)
opt = optim.Adam(vae.parameters(), lr=1e-3)
print(sum(p.numel() for p in vae.parameters()))


In [ ]:
def train_vae_epoch():
    vae.train()
    total = 0.0
    for x, _ in train_loader:
        x = x.to(device).view(x.size(0), -1)
        opt.zero_grad()
        x_recon, z_mean, z_logvar = vae(x)
        loss = loss_fn(x_recon, x, z_mean, z_logvar)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    return total / len(train_loader.dataset)

for e in range(1, epochs + 1):
    l = train_vae_epoch()
    print(e, l)


In [ ]:
vae.eval()
x, _ = next(iter(test_loader))
x = x.to(device)[:16]
x_flat = x.view(x.size(0), -1)
with torch.no_grad():
    x_recon, _, _ = vae(x_flat)
x_recon = x_recon.view(-1, 1, 28, 28).cpu().numpy()
x = x.cpu().numpy()

plt.figure(figsize=(8,4))
for i in range(16):
    plt.subplot(2, 16, i + 1)
    plt.imshow(x[i,0], cmap="gray")
    plt.axis("off")
    plt.subplot(2, 16, 16 + i + 1)
    plt.imshow(x_recon[i,0], cmap="gray")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
vae.eval()
n = 20
img_size = 28
grid = np.zeros((img_size * n, img_size * n))
lin = np.linspace(-2, 2, n)
with torch.no_grad():
    for i, yi in enumerate(lin):
        for j, xi in enumerate(lin):
            z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            x_gen = vae.decoder(z).view(img_size, img_size).cpu().numpy()
            grid[i*img_size:(i+1)*img_size, j*img_size:(j+1)*img_size] = x_gen
plt.figure(figsize=(8,8))
plt.imshow(grid, cmap="gray")
plt.axis("off")
plt.show()


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, pipeline

df_path = None
for p in [Path("spam.csv"), Path("data/text_cls.csv")]:
    if p.exists():
        df_path = p
        break
if df_path is None:
    raise FileNotFoundError("Put dataset into spam.csv or data/text_cls.csv")

if df_path.name.lower() == "spam.csv":
    df = pd.read_csv(df_path, encoding="latin-1")
    df = df.rename(columns={"v1":"label","v2":"text"})
    df = df[["text","label"]]
else:
    df = pd.read_csv(df_path)
    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError("CSV must contain text,label")
    df = df[["text","label"]]

df = df.dropna().reset_index(drop=True)
print(df_path.as_posix(), df.shape)
print(df.head())


In [ ]:
models_root = Path("models")
MODEL_DIR = {
    "spam_1": models_root / "bert-tiny-sms-spam",
    "spam_2": models_root / "distilbert-sms-spam",
    "zs_1": models_root / "xlm-roberta-large-xnli",
    "zs_2": models_root / "mdeberta-v3-base-mnli-xnli",
    "summ": models_root / "uk-summarizer",
    "uk_en": models_root / "opus-mt-uk-en",
    "en_uk": models_root / "opus-mt-en-uk",
}
missing = [k + ":" + v.as_posix() for k, v in MODEL_DIR.items() if not v.exists()]
if missing:
    raise FileNotFoundError("Missing local HF model dirs: " + "; ".join(missing))
print("ok")


In [ ]:
texts = df["text"].astype(str).tolist()
y_true = df["label"].astype(str).str.lower().tolist()
labels = sorted(set(y_true))
use_spam_models = set(labels) == {"ham","spam"} or set(labels) == {"spam","ham"}

device_id = 0 if torch.cuda.is_available() else -1

if use_spam_models:
    tok1 = AutoTokenizer.from_pretrained(str(MODEL_DIR["spam_1"]), local_files_only=True)
    mod1 = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR["spam_1"]), local_files_only=True)
    tok2 = AutoTokenizer.from_pretrained(str(MODEL_DIR["spam_2"]), local_files_only=True)
    mod2 = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR["spam_2"]), local_files_only=True)
    p1 = pipeline("text-classification", model=mod1, tokenizer=tok1, truncation=True, device=device_id)
    p2 = pipeline("text-classification", model=mod2, tokenizer=tok2, truncation=True, device=device_id)

    def norm_spam(l):
        s = str(l).lower()
        if "spam" in s:
            return "spam"
        if "ham" in s or "not_spam" in s or "not spam" in s:
            return "ham"
        if s in {"label_0","0"}:
            return "ham"
        if s in {"label_1","1"}:
            return "spam"
        return s

    r1 = p1(texts, batch_size=32)
    r2 = p2(texts, batch_size=32)
    y1 = [norm_spam(x["label"]) for x in r1]
    y2 = [norm_spam(x["label"]) for x in r2]
else:
    cands = labels
    tok1 = AutoTokenizer.from_pretrained(str(MODEL_DIR["zs_1"]), local_files_only=True)
    mod1 = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR["zs_1"]), local_files_only=True)
    tok2 = AutoTokenizer.from_pretrained(str(MODEL_DIR["zs_2"]), local_files_only=True)
    mod2 = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR["zs_2"]), local_files_only=True)
    p1 = pipeline("zero-shot-classification", model=mod1, tokenizer=tok1, device=device_id)
    p2 = pipeline("zero-shot-classification", model=mod2, tokenizer=tok2, device=device_id)

    def zs_pred(pipe, t):
        o = pipe(t, cands, hypothesis_template="This text is about {}.")
        return o["labels"][0], o

    y1, y2, r1, r2 = [], [], [], []
    for t in texts:
        a, oa = zs_pred(p1, t)
        b, ob = zs_pred(p2, t)
        y1.append(a); r1.append(oa)
        y2.append(b); r2.append(ob)

acc1 = accuracy_score(y_true, y1)
acc2 = accuracy_score(y_true, y2)
print(acc1, acc2)
print(classification_report(y_true, y1))
print(classification_report(y_true, y2))

out = pd.DataFrame({"text": texts, "label_true": y_true, "pred_1": y1, "pred_2": y2})
out.to_csv("out_hf_text_classification_results.csv", index=False, encoding="utf-8")
print(out.head())


In [ ]:
zs_tok = AutoTokenizer.from_pretrained(str(MODEL_DIR["zs_1"]), local_files_only=True)
zs_mod = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR["zs_1"]), local_files_only=True)
zs = pipeline("zero-shot-classification", model=zs_mod, tokenizer=zs_tok, device=device_id)

text_ua = "Компанія впровадила двофакторну автентифікацію та обов'язкове навчання з кібербезпеки для співробітників."
labs = ["кібербезпека", "фінанси", "спорт", "освіта"]
print(zs(text_ua, labs, hypothesis_template="Це текст про {}.")["labels"][:2])

s_tok = AutoTokenizer.from_pretrained(str(MODEL_DIR["summ"]), local_files_only=True)
s_mod = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR["summ"]), local_files_only=True)
summ = pipeline("summarization", model=s_mod, tokenizer=s_tok, device=device_id)

long_ua = "В останні роки цифрові сервіси активно розвиваються, але разом із цим зростають і кіберзагрози. Організації все частіше впроваджують багатофакторну автентифікацію, резервне копіювання та принцип найменших привілеїв. Важливо також навчати користувачів розпізнавати фішингові повідомлення та перевіряти підозрілі посилання. Комплексні заходи безпеки знижують ризик витоку даних і простоїв критичних систем."
print(summ(long_ua, max_length=80, min_length=25, do_sample=False)[0]["summary_text"])

uk_en_tok = AutoTokenizer.from_pretrained(str(MODEL_DIR["uk_en"]), local_files_only=True)
uk_en_mod = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR["uk_en"]), local_files_only=True)
en_uk_tok = AutoTokenizer.from_pretrained(str(MODEL_DIR["en_uk"]), local_files_only=True)
en_uk_mod = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR["en_uk"]), local_files_only=True)

tr_uk_en = pipeline("translation", model=uk_en_mod, tokenizer=uk_en_tok, device=device_id)
tr_en_uk = pipeline("translation", model=en_uk_mod, tokenizer=en_uk_tok, device=device_id)

print(tr_uk_en("Мені потрібен короткий переклад цього речення на англійську.")[0]["translation_text"])
print(tr_en_uk("I need a short translation of this sentence into Ukrainian.")[0]["translation_text"])
